In [0]:
library(dataiku)
library(rpart)
library(dplyr)
library(caret)
library(data.table)
library(mlflow)
library(reticulate)
library(Metrics)

In [0]:
# Recipe inputs
ass_XGBOOST_base_reg <- dkuManagedFolderPath("DFdq9elg")
ass_XGBOOST_classifier <- dkuManagedFolderPath("fZ8zhmA4")
ass_XGBOOST_trunc_reg <- dkuManagedFolderPath("ZGOs5kwX")


# Construct the file path for the model
base_reg_file_path <- file.path(ass_XGBOOST_base_reg, "damage_fit_reg_base.rds")
trunc_reg_file_path <- file.path(ass_XGBOOST_trunc_reg, "trunc_damage_fit_reg.rds")
clas_file_path <- file.path(ass_XGBOOST_classifier, "ass_XGBOOST_class.rds")

# Read the .rds models
base_reg <- readRDS(base_reg_file_path)
trunc_reg <- readRDS(trunc_reg_file_path)
clas_model  <- readRDS(clas_file_path)


In [0]:
# HURDLE METHOD FUNCTION
#' Title: Predict the building damage % from TCs
#'
#' Function takes the test data & trained models and returns predicted building damages.
#'
#' @param df A dataframe for prediction (can be the test set for testing hurdle method)
#' @param class_model The trained model for classification
#' @param scm_models_base A list of the SCM models for the base regression
#' @param scm_models_high A list of SCM models for the high-impact regression
#'
#'

predictDamage <- function(df, ass, scm_models_high, threshold) {

base_col_models_list <- list(
  #track_min_dist = scm_models_base[["base_track_model"]],
  wind_max = scm_models_base[["base_wind_model"]],
  rain_total = scm_models_base[["base_rain_model"]]
  #roof_strong_wall_strong = scm_models_base[["base_roof_strong_wall_strong_model"]],
  #roof_strong_wall_light = scm_models_base[["base_roof_strong_wall_light_model"]],
  #roof_strong_wall_salv = scm_models_base[["base_roof_strong_wall_salv_model"]],
  #roof_light_wall_strong = scm_models_base[["base_roof_light_wall_strong_model"]],
  #roof_light_wall_light = scm_models_base[["base_roof_light_wall_light_model"]],
  #roof_light_wall_salv = scm_models_base[["base_roof_light_wall_salv_model"]],
  #roof_salv_wall_strong = scm_models_base[["base_roof_salv_wall_strong_model"]],
  #roof_salv_wall_light = scm_models_base[["base_roof_salv_wall_light_model"]],
  #roof_salv_wall_salv = scm_models_base[["base_roof_salv_wall_salv_model"]]
)

  ## common predictions btw class & base regression
  df <-  df %>%
  mutate(across(names(base_col_models_list), ~ predict(base_col_models_list[[cur_column()]],
                                             newdata = df), .names = "{.col}_pred"))
  # factors cleaning for classification task
  df$damage_binary_2 <- factor(df$damage_binary,
                                       levels = c("0", "1"),  # Your current levels
                                       labels = c("Damage_below_10", "Damage_above_10"))  # New valid labels

  ## Step 1: Predict the class label (whether the damage will exceed the threshold)
  ## class_model should return predicted classes and not probs.
  ## class_model expects variables "wind_max_pred" and "rain_total_pred" in dataframe df
  ## type = "prob" for custom threshold specification
  prob_pred <- predict(scm_models_base$base_clas_full_model, df, type = "prob")[,2]  # Probability of class 1
  ## assigning final class based on threshold
  class_pred <- ifelse(prob_pred > threshold, 1, 0) # low threhold of 0.35 can be changed to 0.65/0.75

  class_pred  <- factor(class_pred, levels = c("0", "1"),  # Your current levels
                                       labels = c("Damage_below_10", "Damage_above_10"))  # New valid labels

  ## Step 2: Predict the base damage percentage using the base regression model (for low impact cases)
  ## base expects variables "wind_max_pred" and "rain_total_pred" in dataframe df
  ## should return the predicted damage percentages
  base_pred <- predict(scm_models_base$base_reg_model, df)

  ## Step 3: Predict the high-impact damage percentage using the high-impact
  ### SCM models (for high impact cases)
  ## wind and rainfall predictions are based on high impact data (damage >= 10)

  trunc_col_models_list <- list(
  #track_min_dist = scm_models_high[["trunc_track_model"]],
  wind_max = scm_models_high[["trunc_wind_model"]],
  rain_total = scm_models_high[["trunc_rain_model"]]
  #roof_strong_wall_strong = scm_models_high[["trunc_roof_strong_wall_strong_model"]],
  #roof_strong_wall_light = scm_models_high[["trunc_roof_strong_wall_light_model"]],
  #roof_strong_wall_salv = scm_models_high[["trunc_roof_strong_wall_salv_model"]],
  #roof_light_wall_strong = scm_models_high[["trunc_roof_light_wall_strong_model"]],
  #roof_light_wall_light = scm_models_high[["trunc_roof_light_wall_light_model"]],
  #roof_light_wall_salv = scm_models_high[["trunc_roof_light_wall_salv_model"]],
  #roof_salv_wall_strong = scm_models_high[["trunc_roof_salv_wall_strong_model"]],
  #roof_salv_wall_light = scm_models_high[["trunc_roof_salv_wall_light_model"]],
  #roof_salv_wall_salv = scm_models_high[["trunc_roof_salv_wall_salv_model"]]
)
  # add the predictions of wind and rainfall to the dataframe df
  df2 <- df %>%
      mutate(across(names(trunc_col_models_list), ~ predict(trunc_col_models_list[[cur_column()]],
                                             newdata = df), .names = "{.col}_pred"))

  high_pred <- predict(scm_models_high$trunc_reg_model, df2)

  # Step 4: Apply the hurdle method logic
  predicted_damage <- ifelse(class_pred == "Damage_above_10", high_pred, base_pred)

  # Return the predicted damage
  return(predicted_damage)
}

# -------------------------------------------------------------------------------- NOTEBOOK-CELL: CODE
names(base_models_list)

# -------------------------------------------------------------------------------- NOTEBOOK-CELL: CODE
# predicting on base test set data
## because we already implemented the hurdle method
df_test <- bind_rows(
  base_test,
  truncated_test
)

# setting threshold for classification step
threshold = 0.35

preds <- predictDamage(df = df_test, scm_models_base = base_models_list,
  scm_models_high = trunc_models_list, threshold = threshold

)


In [0]:
# Recipe outputs
ass_hurdle_predictions <- dkuManagedFolderPath("K9arRZ6K")